In [1]:
import logging
import os
import argparse
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from data_loader import DataLoader
from common import read_csv, LossEvaluator, COND_OUTCOME_STR, COND_COV_STR
from common import read_groupings
from common import RevisedModel
from eval_model_fixes import parse_proposed_vars, parse_comp_vars

np.random.seed(1)

In [2]:
SPLIT_RATIO = 0.5
NUM_OBS_MDL_REVISE = 2000

def get_bootstrap_metric(Y_true, Y_pred, Y_prob, n_bootstrap=1000, alpha=0.05, rng_seed=0):
    rng = np.random.RandomState(rng_seed)
    indices = rng.randint(0, len(Y_true), (n_bootstrap, len(Y_true)))
    Y_true_boot = Y_true[indices]
    Y_pred_boot = Y_pred[indices]
    Y_prob_boot = Y_prob[indices]
    
    acc = np.mean(Y_true_boot == Y_pred_boot, axis=1)
    auc = np.array([roc_auc_score(Y_true_boot[i], Y_prob_boot[i]) for i in range(n_bootstrap)])
    
    results = {
        'acc': np.quantile(acc, q=[alpha/2, 0.5, 1-alpha/2]),
        'auc': np.quantile(auc, q=[alpha/2, 0.5, 1-alpha/2])
    }
    return results

def get_acc_auc(model, X, Y):
    Y = Y.flatten().astype(int)
    pred_prob = model.predict_proba(X)[:,1]
    pred_Y = model.predict(X).flatten().astype(int)
    assert len(X)==len(Y), "X, Y should be of same shape"
    conf_intervals = get_bootstrap_metric(Y, pred_Y, pred_prob, n_bootstrap=1000, alpha=0.05)  # format is (alpha/2, 0.5, 1-alpha/2) quartiles
    performances = {
        'acc': [(pred_Y == Y).mean()],
        'acc_lower': [conf_intervals['acc'][0]],
        'acc_upper': [conf_intervals['acc'][2]],
        'auc': [roc_auc_score(Y, pred_prob)],
        'auc_lower': [conf_intervals['auc'][0]],
        'auc_upper': [conf_intervals['auc'][2]],
    }  # each value is a list so it in can be converted to dataframe
    return performances

def retrain_model_with_top_explanations(feature_values, vars_names, ml_mdl, trainX, trainY, testX, testY, test_mask, revised_mdl_file, num_top_feats=1):
    # Parse variable names
    top_idxs = np.argsort(feature_values)[-num_top_feats:]  # highest feature value
    top_vars = set(np.concatenate([vars_names[i] for i in top_idxs]))  # variables for fine-tuning
    top_vars = sorted(top_vars)
    train_pred_prob = ml_mdl.predict_proba(trainX)[:,1:]
    print("TOP FEATS", top_vars)
    retrain_feats = np.concatenate([
        # train_pred_prob,
        np.log(train_pred_prob/(1 - train_pred_prob)),
        trainX[:,top_vars],
    ], axis=1)

    revised_mdl = clone(ml_mdl)
    revised_mdl.fit(retrain_feats, trainY)
    with open(revised_mdl_file, "wb") as f:
        revised_mdl_save = RevisedModel(
            org_mdl=ml_mdl,
            revised_mdl=revised_mdl,
            vars_mask=top_vars
        )
        pickle.dump(revised_mdl_save, f)

    performances = {'test_set': [], 'acc': [], 'acc_lower': [], 'acc_upper': [], 'auc': [], 'auc_lower': [], 'auc_upper': []}
    testX_list = [testX, testX[test_mask]]
    true_testY_list = [testY, testY[test_mask]]
    test_descr_list = ['all', 'subset']
    for testX, true_testY, test_descr in zip(testX_list, true_testY_list, test_descr_list):
        print(len(testX), len(true_testY), test_descr)
        test_pred_prob = ml_mdl.predict_proba(testX)[:,1:]
        retest_feats = np.concatenate([
            # test_pred_prob,
            np.log(test_pred_prob/(1 - test_pred_prob)),
            testX[:, top_vars],
        ], axis=1)
        true_test_Y = true_testY.flatten().astype(int)
        # pred_prob = revised_mdl.predict_proba(retest_feats)[:,1]
        # log_lik = np.mean(true_test_Y * np.log(pred_prob) + (1 - true_test_Y) * np.log(1-pred_prob))
        test_perf = get_acc_auc(revised_mdl, retest_feats, true_test_Y)
        for m in test_perf:
            performances[m] += test_perf[m]
        performances['test_set'].append(test_descr)
    return performances

# Update as per TEVIM features

In [ ]:
explainer_data = '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/casestudy/_output/acs_pubcov_save_scaled/MLPClassifier/accuracy/0.05/40/explainer1.pkl'
loss = 'accuracy'
decomposition = 'Cond_Outcome'
tolerance = 0.05
result_data = '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/casestudy/_output/acs_pubcov_save_scaled/MLPClassifier/accuracy/0.05/40/comp_resultestimate1.csv'
grouping_data = "/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/data/acs_pubcov_feature_groupings.csv"
train_data = "/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/data/acs_pubcov_source_train.csv"
source_data = "/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/data/acs_pubcov_source_val.csv"
target_data = "/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/data/acs_pubcov_target_modelfix.csv"  # target data unused by TE-VIM and SHIFT
mdl_file = '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/casestudy/_output/acs_pubcov_save_scaled/MLPClassifier/mdl.pkl'
revised_mdl_targeted_file = '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/casestudy/_output/acs_pubcov_save_scaled/MLPClassifier/tevim_revised_mdl_targeted.pkl'

In [4]:
sourceX, sourceY = read_csv(source_data)
sourceX, sourceY = sourceX.to_numpy(), sourceY.to_numpy()
targetX, targetY = read_csv(target_data)
targetX, targetY = targetX.to_numpy(), targetY.to_numpy()
variable_dict = read_groupings(grouping_data)

source_loader = DataLoader(sourceX, sourceY, variable_dict)
# target_loader = DataLoader(targetX, targetY, variable_dict)

train_sourceX, train_sourceY = sourceX, sourceY
train_targetX, test_targetX, train_targetY, test_targetY = train_test_split(
    targetX, targetY, test_size = SPLIT_RATIO, random_state=1
)

train_targetX = train_targetX[:NUM_OBS_MDL_REVISE]
train_targetY = train_targetY[:NUM_OBS_MDL_REVISE]

with open(mdl_file, "rb") as f:
        ml_mdl = pickle.load(f)

X    SEX  AGEP  DIS  ESP  MIG  MIL  ANC  NATIVITY  DEAR  DEYE  ...  CIT_pr  \
0  1.0  15.0  2.0  1.0  1.0  0.0  2.0       1.0   2.0   2.0  ...     0.0   
1  2.0  45.0  2.0  0.0  1.0  4.0  1.0       1.0   2.0   2.0  ...     0.0   
2  2.0  40.0  2.0  0.0  1.0  4.0  2.0       1.0   2.0   2.0  ...     0.0   
3  2.0  46.0  2.0  0.0  1.0  4.0  4.0       1.0   2.0   2.0  ...     0.0   
4  2.0  61.0  2.0  0.0  1.0  4.0  1.0       1.0   2.0   2.0  ...     0.0   

   CIT_abroad  CIT_citizen  CIT_not  ESR_employed  ESR_partial_employed  \
0         0.0          0.0      0.0           0.0                   0.0   
1         0.0          0.0      0.0           1.0                   0.0   
2         0.0          0.0      0.0           1.0                   0.0   
3         0.0          0.0      0.0           0.0                   1.0   
4         0.0          0.0      0.0           1.0                   0.0   

   ESR_unemployed  ESR_armed  ESR_partial_armed  ESR_no  
0             0.0        0.0    

In [5]:
with open(mdl_file, "rb") as f:
        ml_mdl = pickle.load(f)

In [6]:
pd.read_csv(result_data)

,value,level,decomp,vars,est,pvalue,job,mdl,nsource,ntarget
0,0.432564,detail,Cond_Outcome,X1,ParametricChangeExplanation,0.146266,1,casestudy/_output/acs_pubcov_save_scaled/MLPCl...,3166,12000
1,0.165448,detail,Cond_Outcome,X2,ParametricChangeExplanation,0.378328,1,casestudy/_output/acs_pubcov_save_scaled/MLPCl...,3166,12000
2,-0.003268,detail,Cond_Outcome,X3,ParametricChangeExplanation,0.591505,1,casestudy/_output/acs_pubcov_save_scaled/MLPCl...,3166,12000
3,0.246611,detail,Cond_Outcome,X4,ParametricChangeExplanation,0.236053,1,casestudy/_output/acs_pubcov_save_scaled/MLPCl...,3166,12000
4,0.006381,detail,Cond_Outcome,X5,ParametricChangeExplanation,0.894438,1,casestudy/_output/acs_pubcov_save_scaled/MLPCl...,3166,12000
...,...,...,...,...,...,...,...,...,...,...
73,0.001561,detail,Cond_Outcome,"X3,X4,X5,X6,X9,X10,X11,X12,X13,X20,X21,X22,X23...",TEVIMTest,0.013000,1,casestudy/_output/acs_pubcov_save_scaled/MLPCl...,3166,12000
74,0.001992,detail,Cond_Outcome,"X1,X2,X4,X5,X6,X7,X8,X12,X13,X14,X15,X16,X17,X...",TEVIMTest,0.003000,1,casestudy/_output/acs_pubcov_save_scaled/MLPCl...,3166,12000
75,0.002027,detail,Cond_Outcome,"X1,X2,X3,X7,X8,X9,X10,X11,X12,X14,X15,X16,X17,...",TEVIMTest,0.007000,1,casestudy/_output/acs_pubcov_save_scaled/MLPCl...,3166,12000
76,0.000961,detail,Cond_Outcome,"X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,X13,X14,X15...",TEVIMTest,0.079000,1,casestudy/_output/acs_pubcov_save_scaled/MLPCl...,3166,12000


In [7]:
df_proposed = pd.read_csv(result_data)
df_proposed = df_proposed[df_proposed.est == "TEVIMTest"]
feat_vals = df_proposed.pvalue[df_proposed.level == "detail"].to_numpy()
vars_names_str = df_proposed.vars[df_proposed.level == "detail"].to_numpy()
vars_names = [parse_comp_vars(v) for v in vars_names_str]
complement_vars_names = [[i for i in range(train_targetX.shape[1]) if i not in v] for v in vars_names]
feat_vals

array([0.013, 0.003, 0.007, 0.079, 0.   ])

In [8]:
complement_vars_names

[[0, 1, 6, 7, 13, 14, 15, 16, 17, 18, 23, 24, 25, 26, 27],
 [2, 8, 9, 10],
 [3, 4, 5, 12],
 [11, 28, 29, 30, 31, 32, 33],
 [19, 20, 21, 22]]

In [9]:
with open(explainer_data, "rb") as f:
        detectors = pickle.load(f)

In [10]:
detector_y = detectors['agg_detectors_y'][0]

detection_on_train_targetX = detector_y.predict(train_targetX)
detection_on_test_targetX = detector_y.predict(test_targetX)
train_target_mask = detection_on_train_targetX > 0
test_target_mask = detection_on_test_targetX > 0

train_target_mask.mean()

0.5075

In [11]:
subgroup_train_targetX = train_targetX[train_target_mask]
subgroup_train_targetY = train_targetY[train_target_mask]
subgroup_test_targetX = test_targetX[test_target_mask]
subgroup_test_targetY = test_targetY[test_target_mask]

print(len(subgroup_train_targetX), len(subgroup_train_targetX) / len(train_targetX))

1015 0.5075


In [12]:
train_targetX.shape

(2000, 34)

In [13]:
subgroup_train_targetX.shape

(1015, 34)

In [14]:
# Revise model on top feature group, test on detected subgroup
num_top = 1
acc_targeted = retrain_model_with_top_explanations(
        feat_vals,
        complement_vars_names,
        ml_mdl,
        train_targetX,
        train_targetY,
        test_targetX,
        test_targetY,
        test_target_mask,
        revised_mdl_targeted_file,
        num_top_feats=num_top,
    )
print("TARGETED: ACC, AUC on target SUBGROUP\n", (pd.DataFrame(acc_targeted) * 100).round(1))

TOP FEATS [11, 28, 29, 30, 31, 32, 33]
2440 2440 all
1228 1228 subset
TARGETED: ACC, AUC on target SUBGROUP
                                             test_set   acc  acc_lower  \
0  allallallallallallallallallallallallallallalla...  68.9       67.1   
1  subsetsubsetsubsetsubsetsubsetsubsetsubsetsubs...  66.1       63.4   

   acc_upper   auc  auc_lower  auc_upper  
0       70.7  73.0       71.0       75.1  
1       68.6  64.9       61.6       67.9  
